# Actividad 4 - Método no supervisado (K-Means)

En esta actividad se aplica un análisis de clustering sobre un dataset de paradas de transporte público. El objetivo es identificar grupos de paradas con comportamientos similares a partir de variables como pasajeros promedio, tiempo de espera, velocidad y transbordos.

## Variables del dataset
- `pasajeros_promedio`: cantidad de pasajeros por hora
- `tiempo_espera_min`: tiempo promedio de espera en minutos
- `velocidad_promedio_kmh`: velocidad media del tramo
- `num_transbordos_frecuentes`: frecuencia de transbordos

## Flujo del análisis
1. Cargar el dataset existente
2. Escalar las variables con `StandardScaler`
3. Aplicar método del codo
4. Calcular el coeficiente de silueta
5. Elegir el número de clusters
6. Entrenar `KMeans`
7. Interpretar los centroides
8. Visualizar los resultados


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Carga del dataset existente
df = pd.read_csv('dataset_no_supervisado.csv')

features = [
    'pasajeros_promedio',
    'tiempo_espera_min',
    'velocidad_promedio_kmh',
    'num_transbordos_frecuentes'
]

X = df[features].copy()

# Escalado de variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Evaluación del número de clusters
ks = range(2, 11)
inertia = []
silhouette_values = []

for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_scaled)
    inertia.append(model.inertia_)
    silhouette_values.append(silhouette_score(X_scaled, labels))

print('Inercia por k:')
for k, value in zip(ks, inertia):
    print(f'k={k}: {value:.2f}')

print('
Silhouette Score por k:')
for k, value in zip(ks, silhouette_values):
    print(f'k={k}: {value:.4f}')

best_k_silhouette = ks[np.argmax(silhouette_values)]
print(f'
Mejor k por silueta: {best_k_silhouette}')


## Interpretación del codo y la silueta

El método del codo busca identificar el punto en el que la curva de inercia deja de disminuir de forma brusca. En este caso, la caída se vuelve más suave alrededor de 5 o 6 clusters, lo que sugiere que ese rango es un punto razonable.

Por otra parte, el coeficiente de silueta mide qué tan bien están definidos los grupos. El mejor valor se observa en `k = 8`, pero la mejora respecto a `k = 6` es pequeña y, además, 8 grupos producen una segmentación más fragmentada, menos interpretable.

Por eso, en este caso se elige `k = 6` como número final de clusters: equilibra calidad, interpretabilidad y estabilidad del modelo.


In [ ]:
# Cluster final elegido
k_final = 6

model = KMeans(n_clusters=k_final, random_state=42, n_init=30)
labels = model.fit_predict(X_scaled)

# Centroides en unidades reales
centroids_original = scaler.inverse_transform(model.cluster_centers_)
centroid_df = pd.DataFrame(centroids_original, columns=features)
centroid_df.insert(0, 'cluster', range(k_final))

print('Centroides en unidades reales:')
print(centroid_df.round(2).to_string(index=False))

summary = df.copy()
summary['cluster'] = labels
print('
Perfil promedio por cluster:')
print(summary.groupby('cluster')[features].mean().round(2).to_string())


## Interpretación de los clusters

Los centroides permiten resumir cada grupo de paradas:
- `Cluster 0` o `Cluster 1`: paradas con mayor flujo y mayor tiempo de espera, por lo que se pueden considerar zonas de congestión o alta demanda.
- `Cluster 2` o `Cluster 3`: paradas con flujo medio y tiempos moderados, sugerentes de operación estable.
- `Cluster 4` o `Cluster 5`: paradas con menor afluencia, menor espera y mayor rapidez, adecuadas para zonas de menor presión operativa.

Esto permite priorizar intervenciones en puntos críticos del sistema, como mayor frecuencia de buses o ajustes de rutas.


In [ ]:
# Gráfico 1: Método del codo
plt.figure(figsize=(8, 5))
plt.plot(list(ks), inertia, marker='o', linewidth=2, color='royalblue')
plt.title('Método del Codo (Elbow Method)')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia dentro del cluster')
plt.grid(True, linestyle='--', alpha=0.4)
plt.xticks(list(ks))
plt.tight_layout()
plt.show()

# Gráfico 2: Coeficiente de silueta
plt.figure(figsize=(8, 5))
plt.plot(list(ks), silhouette_values, marker='o', linewidth=2, color='darkgreen')
plt.axvline(best_k_silhouette, color='red', linestyle='--', label=f'Mejor k por silueta: {best_k_silhouette}')
plt.title('Coeficiente de Silueta por número de clusters')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--', alpha=0.4)
plt.xticks(list(ks))
plt.legend()
plt.tight_layout()
plt.show()

# Gráfico 3: Agrupamiento en dos dimensiones (PCA)
reducer = PCA(n_components=2)
X_pca = reducer.fit_transform(X_scaled)
centroid_pca = reducer.transform(model.cluster_centers_)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', s=45, alpha=0.75)
plt.scatter(centroid_pca[:, 0], centroid_pca[:, 1], c='red', marker='X', s=180, linewidths=1.5, label='Centroides')
plt.title('Agrupamiento K-Means (PCA 2D)')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.tight_layout()
plt.show()
